# 13.02 · Inference engines

**Chapter:** Serving & Deployment · **Milestone:** M5

## Goals
- [ ] Explain the two phases of inference: **prefill** (compute-bound) and **decode** (memory-bandwidth-bound)
- [ ] KV cache: why it exists, how big it gets, and how PagedAttention / GQA / KV quantization tame it
- [ ] Batching: static vs **continuous** batching, and why throughput and latency trade off
- [ ] Speculative decoding: a small draft model proposes, the big model verifies — same output, faster
- [ ] Pick an engine: vLLM vs SGLang vs TensorRT-LLM vs llama.cpp/Ollama vs MLX-LM

## Resources
- [vLLM docs](https://docs.vllm.ai/) · [PagedAttention paper](https://arxiv.org/abs/2309.06180) — free
- [SGLang docs](https://docs.sglang.ai/) (RadixAttention prefix caching) — free
- [HF – Assisted generation (speculative decoding)](https://huggingface.co/blog/assisted-generation) — free
- [Lilian Weng – Large Transformer Model Inference Optimization](https://lilianweng.github.io/posts/2023-01-10-inference-optimization/) — free
- [Baseten / Modal – LLM inference guides](https://modal.com/llm-almanac) — free

In [ ]:
import subprocess
import sys
import threading
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from aieng import DATA
from aieng.llm import client

device = "mps" if torch.backends.mps.is_available() else "cpu"
BASE = "HuggingFaceTB/SmolLM2-135M-Instruct"
tok = AutoTokenizer.from_pretrained(BASE)
model = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.float32).to(device).eval()

## 1. Prefill vs decode
**Prefill** runs the whole prompt through the model in one parallel pass (big matrix multiplies: compute-bound) and produces the first token.
**Decode** then generates one token at a time; each step does little math but must read every weight (memory-bound).
That is why we track **TTFT** (time to first token ≈ prefill) separately from **tokens/sec** (decode).

In [ ]:
def timed_generate(prompt_tokens, new_tokens, **kw):
    x = torch.randint(100, 10_000, (1, prompt_tokens), device=device)
    if device == "mps": torch.mps.synchronize()
    t = time.perf_counter()
    model.generate(x, max_new_tokens=new_tokens, min_new_tokens=new_tokens, do_sample=False, pad_token_id=tok.eos_token_id, **kw)
    if device == "mps": torch.mps.synchronize()
    return time.perf_counter() - t


timed_generate(16, 4)  # warm-up
for n in (128, 512, 2048):
    ttft = timed_generate(n, 1)
    print(f"prompt {n:5} tokens: prefill {ttft * 1000:6.1f} ms ({n / ttft:8,.0f} tok/s)")
decode = (timed_generate(128, 129) - timed_generate(128, 1)) / 128
print(f"decode: {decode * 1000:.1f} ms/token ({1 / decode:,.0f} tok/s)")

Prefill processes thousands of tokens/sec; decode tens to hundreds. Long prompts cost TTFT; long answers cost decode time.

## 2. The KV cache
Attention at step *t* needs keys/values of all previous tokens. Without a cache, every step recomputes them for the whole sequence (O(n²) total);
with a **KV cache** each step only computes the new token's K/V and appends them.

In [ ]:
for use_cache in (False, True):
    secs = timed_generate(64, 256, use_cache=use_cache)
    print(f"use_cache={use_cache!s:5}: 256 tokens in {secs:5.2f}s")

The price is memory: `2 (K and V) × layers × kv_heads × head_dim × bytes × tokens`, **per sequence**.

In [ ]:
def kv_gb(layers, kv_heads, head_dim, tokens, batch=1, bytes_=2):
    return 2 * layers * kv_heads * head_dim * bytes_ * tokens * batch / 1e9


cfg = model.config
print(f"SmolLM2-135M, 8k ctx: {kv_gb(cfg.num_hidden_layers, cfg.num_key_value_heads, cfg.hidden_size // cfg.num_attention_heads, 8192):.2f} GB")
print(f"Llama-3.1-8B (GQA, 8 kv heads), 8k ctx, batch 32: {kv_gb(32, 8, 128, 8192, 32):.0f} GB   <- more than the 16 GB of weights")
print(f"same model without GQA (32 kv heads):           {kv_gb(32, 32, 128, 8192, 32):.0f} GB")
print(f"with FP8 KV cache (GQA):                         {kv_gb(32, 8, 128, 8192, 32, bytes_=1):.0f} GB")

At high concurrency the KV cache, not the weights, limits how many users a GPU serves. The fixes:
- **GQA / MQA** (in the architecture): several query heads share one K/V head → 4–8× smaller cache.
- **PagedAttention** (vLLM): store KV in fixed-size blocks like OS virtual memory: no fragmentation, no over-reserving max length.
- **Prefix caching** (vLLM automatic prefix caching, SGLang RadixAttention): requests sharing a prefix (system prompt, few-shot, RAG docs) reuse its KV.
  This is what provider "prompt caching" discounts are (05.05).
- **KV quantization** (FP8/int8 KV) and sliding-window / sparse attention for very long contexts.

## 3. Batching and throughput
Decode is memory-bound, so processing 8 sequences per step costs barely more than 1: the weights are read once for all of them.

In [ ]:
prompts = [f"Write a short poem about the number {i}." for i in range(8)]
tok.padding_side = "left"
for bs in (1, 8):
    batch = tok([tok.apply_chat_template([{"role": "user", "content": p}], tokenize=False, add_generation_prompt=True) for p in prompts[:bs]],
                return_tensors="pt", padding=True).to(device)
    t = time.perf_counter()
    model.generate(**batch, max_new_tokens=128, min_new_tokens=128, do_sample=False, pad_token_id=tok.eos_token_id)
    secs = time.perf_counter() - t
    print(f"batch {bs}: {secs:5.2f}s -> {bs * 128 / secs:6.0f} tokens/s total, {128 / secs:5.0f} tokens/s per sequence")

**Static batching** waits for the whole batch to finish (short answers idle while long ones finish). **Continuous batching** (vLLM, SGLang, TGI,
llama.cpp server, Ollama with `OLLAMA_NUM_PARALLEL`) adds and removes sequences *every step*, keeping the GPU full. Throughput goes up with
concurrency while per-user speed goes down a little: the core serving trade-off.

Measure it on Ollama with concurrent requests:

In [ ]:
oc, MODEL = client("ollama")


def one_request(results):
    t = time.perf_counter()
    r = oc.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": "Count from 1 to 40, comma separated."}],
                                   max_tokens=150, temperature=0, reasoning_effort="none")
    results.append((r.usage.completion_tokens, time.perf_counter() - t))


one_request([])  # load the model
rows = []
for n in (1, 2, 4, 8):
    results, t = [], time.perf_counter()
    threads = [threading.Thread(target=one_request, args=(results,)) for _ in range(n)]
    [th.start() for th in threads]; [th.join() for th in threads]
    wall = time.perf_counter() - t
    toks = sum(r[0] for r in results)
    rows.append((n, toks / wall, np.mean([r[0] / r[1] for r in results])))
    print(f"{n} concurrent: {toks / wall:6.0f} tok/s total | {rows[-1][2]:5.0f} tok/s per request | wall {wall:.1f}s")

In [ ]:
n, total, per = zip(*rows)
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(n, total, "o-", label="total throughput"); ax.plot(n, per, "s--", label="per-request speed")
ax.set_xlabel("concurrent requests"); ax.set_ylabel("tokens/s"); ax.legend(); plt.show()

If total throughput stays flat, Ollama is serving requests one at a time: set `OLLAMA_NUM_PARALLEL=4` (e.g.
`launchctl setenv OLLAMA_NUM_PARALLEL 4` then restart Ollama) and re-run. Production engines on a GPU scale to hundreds of concurrent sequences.

## 4. Speculative decoding
A small **draft** model guesses the next *k* tokens; the big **target** model checks all *k* in one forward pass (cheap: like prefill) and keeps
the longest correct prefix. With greedy decoding the output is mathematically the same as the target alone — only faster when the draft guesses well.
Variants: EAGLE / Medusa (draft heads inside the model), n-gram / prompt lookup (copy from the prompt: great for RAG and code edits).

MLX-LM supports it directly. Target: Qwen3-1.7B 4-bit (cached since 12.03); draft: the Qwen3-0.6B 4-bit you made in 13.01 (same tokenizer).

In [ ]:
from mlx_lm import load, stream_generate

target, ttok = load("mlx-community/Qwen3-1.7B-4bit")
draft_path = DATA / "mlx-quant" / "qwen3-0.6b-4"
assert draft_path.exists(), "Run 13.01 first (it creates the draft model)."
draft, _ = load(str(draft_path))
prompt = ttok.apply_chat_template([{"role": "user", "content": "List the planets of the solar system with one fact each."}],
                                  add_generation_prompt=True, enable_thinking=False)


def run(**kw):
    text, accepted = "", 0
    for r in stream_generate(target, ttok, prompt, max_tokens=200, **kw):
        text += r.text; accepted += bool(getattr(r, "from_draft", False))
    return text, r.generation_tps, accepted / max(r.generation_tokens, 1)


plain, tps_plain, _ = run()
spec, tps_spec, acc = run(draft_model=draft, num_draft_tokens=3)
print(f"target alone: {tps_plain:5.0f} tok/s")
print(f"speculative : {tps_spec:5.0f} tok/s, {acc:.0%} of tokens came from the draft")
same = next((i for i, (a, b) in enumerate(zip(plain, spec)) if a != b), min(len(plain), len(spec)))
print(f"outputs agree on the first {same} of {len(plain)} characters")

Two honest lessons here:
- **Speed-up needs a much cheaper draft.** Every draft token costs a 0.6B forward pass and rejected guesses are wasted work. At 1.7B vs 0.6B
  the draft is too expensive relative to the target, so speculation can be *slower*. Datacenter pairs (1B draft for a 70B target, or EAGLE heads
  that cost ~nothing) get 2–3×. Always benchmark on your hardware before enabling it.
- **"Identical" holds in exact arithmetic only.** Verifying 3 tokens in one batched pass rounds floats differently than one-at-a-time decoding,
  so at a near-tie between two tokens greedy picks can flip and the texts diverge from there (both are valid greedy outputs of the model).
  Same reason batch size or GPU type can change outputs at temperature 0.

## 5. Every engine speaks the OpenAI API
Start MLX-LM's server, call it with the same client code as Ollama or Groq, stop it.

In [ ]:
server = subprocess.Popen([sys.executable, "-m", "mlx_lm", "server", "--model", "mlx-community/Qwen3-1.7B-4bit", "--port", "8899"],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
from openai import OpenAI

mlx_client = OpenAI(base_url="http://localhost:8899/v1", api_key="none")
try:
    for _ in range(60):
        try:
            mlx_client.models.list(); break
        except Exception:
            time.sleep(1)
    r = mlx_client.chat.completions.create(model="mlx-community/Qwen3-1.7B-4bit", max_tokens=60,
                                           messages=[{"role": "user", "content": "/no_think In one sentence: what is continuous batching in LLM inference servers?"}])
    print(r.choices[0].message.content)
finally:
    server.terminate(); server.wait()

## 6. Choosing an engine

| Engine | Hardware | Strengths | Use when |
|---|---|---|---|
| **vLLM** | NVIDIA, AMD, TPU, CPU | PagedAttention, continuous batching, prefix caching, multi-LoRA, huge model support | default for GPU serving |
| **SGLang** | NVIDIA, AMD | RadixAttention (shared-prefix heavy workloads), fast structured output, strong on agents | many requests sharing prompts |
| **TensorRT-LLM** (+ Triton / NIM) | NVIDIA | compiled kernels, FP8/FP4, best raw latency | max performance on NVIDIA, willing to compile |
| **llama.cpp** / **Ollama** / LM Studio | CPU, Apple, any GPU | GGUF, runs anywhere, easy | local, edge, single-user |
| **MLX-LM** | Apple Silicon | fastest on Mac, LoRA training | Mac local |
| TGI (Hugging Face) | NVIDIA, AMD | HF integration | existing deployments (now in maintenance mode; HF recommends vLLM/SGLang) |

Managed alternatives: Groq/Cerebras (custom chips, very fast), Together, Fireworks, Baseten, Modal, or the cloud (Bedrock, Vertex, Azure).

On a free GPU (Colab/Kaggle T4) vLLM is two commands:
```bash
pip install vllm
vllm serve Qwen/Qwen3-1.7B --max-model-len 8192 --enable-prefix-caching   # OpenAI API on :8000
```

## Exercises
1. On a free Colab T4: serve Qwen3-1.7B with vLLM, rerun §3's concurrency test against it with 1–64 concurrent requests. Compare with Ollama.
2. Enable `OLLAMA_NUM_PARALLEL=4` and redo §3. What happens to per-request speed?
3. Try `num_draft_tokens` 1–6 in §4 and plot tokens/s and acceptance rate. Try a prompt that repeats a lot of its input (e.g. "rewrite this text with
   British spelling: …") — draft acceptance should go up.
4. Compute the KV cache size for a model you'd like to serve at 32k context for 16 users. Does it fit on a 24 GB / 80 GB GPU?

In [ ]:
# your code here